# Chapter 69 — GPT Training Examples: Targets at Every Position

Chapter 68 produced one vocabulary-logit vector at every input position.

This chapter constructs the shifted targets that let all of those positions contribute to next-token training.


## Learning goals

By the end of this chapter, you will be able to:

- explain shifted next-token targets;
- slice one input-target window without an off-by-one error;
- build matching `[B, T]` input and target batches;
- calculate the valid range of start indexes;
- decode and inspect every batch row;
- verify the shift programmatically;
- connect `[B, T, V]` logits to `[B, T]` targets; and
- explain why causal masking is still required.


## One window contains many predictions

A **sequence window** is one fixed-length slice used as a row in a batch.

A **shifted target** uses tokens from the same source text one position later than the input.

For the text `the cat`, a context length of six gives:

```text
source:  t  h  e     c  a  t
input:   t  h  e     c  a
target:  h  e     c  a  t
```

The six aligned prediction tasks are `t → h`, `h → e`, `e → space`, `space → c`, `c → a`, and `a → t`.

A batch with `B` sequence windows contains `B × T` supervised token-position predictions.


## Slice one window

The input starts at source index `s`, while the target starts at `s + 1`.

Both slices must contain exactly `T` tokens.


In [1]:
import torch

device = "cpu"
text = "the cat"
context_length = 6
start_index = 0
input_text = text[start_index : start_index + context_length]
target_text = text[start_index + 1 : start_index + context_length + 1]

print("device:", device)
print("source text:", repr(text))
print("input text: ", repr(input_text))
print("target text:", repr(target_text))
print("matching lengths:", len(input_text) == len(target_text) == context_length)

device: cpu
source text: 'the cat'
input text:  'the ca'
target text: 'he cat'
matching lengths: True


The final `+ 1` in the target slice stop is necessary because Python excludes the stop index.


## Inspect aligned positions

Each context position aligns one visible input token with the next source token as its target.


In [2]:
print("position | source input index | input | source target index | target")
print("-" * 70)
for position in range(context_length):
    input_source_index = start_index + position
    target_source_index = input_source_index + 1
    print(
        f"{position:>8} | "
        f"{input_source_index:>18} | "
        f"{input_text[position]!r:>5} | "
        f"{target_source_index:>19} | "
        f"{target_text[position]!r:>6}"
    )

position | source input index | input | source target index | target
----------------------------------------------------------------------
       0 |                  0 |   't' |                   1 |    'h'
       1 |                  1 |   'h' |                   2 |    'e'
       2 |                  2 |   'e' |                   3 |    ' '
       3 |                  3 |   ' ' |                   4 |    'c'
       4 |                  4 |   'c' |                   5 |    'a'
       5 |                  5 |   'a' |                   6 |    't'


The context position stays the same while the target's source index is always one greater than the input's source index.


## Convert characters to token IDs

A model consumes integer token IDs instead of Python strings.

The encoding preserves order, so the same slicing rule applies to the token tensor.


In [3]:
characters = sorted(set(text))
character_to_id = {
    character: character_id for character_id, character in enumerate(characters)
}
id_to_character = {
    character_id: character for character, character_id in character_to_id.items()
}


def encode_text(source_text: str, token_to_id: dict[str, int]) -> list[int]:
    return [token_to_id[character] for character in source_text]


def decode_token_ids(token_ids: list[int], id_to_token: dict[int, str]) -> str:
    return "".join(id_to_token[token_id] for token_id in token_ids)


token_ids = torch.tensor(
    encode_text(text, character_to_id),
    dtype=torch.long,
    device=device,
)
input_row = token_ids[start_index : start_index + context_length]
target_row = token_ids[start_index + 1 : start_index + context_length + 1]

print("token mapping:", character_to_id)
print("input IDs: ", input_row.tolist())
print("target IDs:", target_row.tolist())
print("decoded input: ", repr(decode_token_ids(input_row.tolist(), id_to_character)))
print("decoded target:", repr(decode_token_ids(target_row.tolist(), id_to_character)))

token mapping: {' ': 0, 'a': 1, 'c': 2, 'e': 3, 'h': 4, 't': 5}
input IDs:  [5, 4, 3, 0, 2, 1]
target IDs: [4, 3, 0, 2, 1, 5]
decoded input:  'the ca'
decoded target: 'he cat'


The integer rows have the same shift and the same length as the earlier string slices.


## Use a longer source sequence

A batch sampler needs enough source tokens to offer many valid windows.

We will build one character vocabulary from the entire training text.


In [4]:
training_text = (
    "the cat sat on the mat. the dog sat on the rug. the cat ran to the dog. "
)
characters = sorted(set(training_text))
character_to_id = {
    character: character_id for character_id, character in enumerate(characters)
}
id_to_character = {
    character_id: character for character, character_id in character_to_id.items()
}
all_token_ids = torch.tensor(
    encode_text(training_text, character_to_id),
    dtype=torch.long,
    device=device,
)

print("source length:", all_token_ids.numel())
print("vocabulary size:", len(characters))
print("characters:", characters)

source length: 72
vocabulary size: 15
characters: [' ', '.', 'a', 'c', 'd', 'e', 'g', 'h', 'm', 'n', 'o', 'r', 's', 't', 'u']


## Find the valid start indexes

Let `N` be the number of source tokens and `T` be context length.

The target needs source index `start_index + T`, so that index must be less than `N`.

Valid starts are `0` through `N - T - 1`, giving `N - T` possible starts.

This is exactly the half-open range accepted by `torch.randint(low=0, high=N - T)`.


In [5]:
context_length = 8
number_of_tokens = all_token_ids.numel()
number_of_valid_starts = number_of_tokens - context_length
largest_valid_start = number_of_valid_starts - 1

last_input = all_token_ids[largest_valid_start : largest_valid_start + context_length]
last_target = all_token_ids[
    largest_valid_start + 1 : largest_valid_start + context_length + 1
]

print("number of tokens:", number_of_tokens)
print("number of valid starts:", number_of_valid_starts)
print("largest valid start:", largest_valid_start)
print("last input length:", last_input.numel())
print("last target length:", last_target.numel())

number of tokens: 72
number of valid starts: 64
largest valid start: 63
last input length: 8
last target length: 8


Even the largest valid start produces full-length input and target rows.


## Build a batch from chosen starts

Separating batch construction from random sampling makes the slicing logic easy to test.

The function below accepts explicit start indexes and returns matching `[B, T]` tensors.


In [6]:
def build_gpt_batch(
    token_ids: torch.Tensor,
    start_indexes: torch.Tensor,
    context_length: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    if token_ids.ndim != 1:
        raise ValueError("token_ids must be one-dimensional.")
    if start_indexes.ndim != 1 or start_indexes.numel() == 0:
        raise ValueError("start_indexes must be a nonempty one-dimensional tensor.")
    if context_length < 1:
        raise ValueError("context_length must be positive.")

    number_of_valid_starts = token_ids.numel() - context_length
    if number_of_valid_starts < 1:
        raise ValueError("token_ids must be longer than context_length.")
    if start_indexes.min().item() < 0:
        raise ValueError("start indexes cannot be negative.")
    if start_indexes.max().item() >= number_of_valid_starts:
        raise ValueError("a start index is outside the valid range.")

    input_rows = []
    target_rows = []
    for start_index in start_indexes.tolist():
        input_rows.append(token_ids[start_index : start_index + context_length])
        target_rows.append(
            token_ids[start_index + 1 : start_index + context_length + 1]
        )

    return torch.stack(input_rows), torch.stack(target_rows)


chosen_start_indexes = torch.tensor([0, 10], device=device)
input_batch, target_batch = build_gpt_batch(
    all_token_ids, chosen_start_indexes, context_length
)

print("input shape:", input_batch.shape)
print("target shape:", target_batch.shape)
print("batch | start | input | target")
print("-" * 54)
for batch_index, start in enumerate(chosen_start_indexes.tolist()):
    decoded_input = decode_token_ids(input_batch[batch_index].tolist(), id_to_character)
    decoded_target = decode_token_ids(
        target_batch[batch_index].tolist(), id_to_character
    )
    print(
        f"{batch_index:>5} | {start:>5} | "
        f"{decoded_input!r:>12} | {decoded_target!r:>12}"
    )

input shape: torch.Size([2, 8])
target shape: torch.Size([2, 8])
batch | start | input | target
------------------------------------------------------
    0 |     0 |   'the cat ' |   'he cat s'
    1 |    10 |   't on the' |   ' on the '


Both rows visibly pair each input substring with the substring that begins one source token later.


## Verify the shift invariant

Within every row, `target_batch[:, :-1]` must equal `input_batch[:, 1:]`.

The final target is the one additional future token that is not present in the input row.


In [7]:
local_shift_matches = torch.equal(target_batch[:, :-1], input_batch[:, 1:])
expected_final_targets = all_token_ids[chosen_start_indexes + context_length]
final_targets_match_source = torch.equal(target_batch[:, -1], expected_final_targets)

print("target prefixes equal shifted inputs:", local_shift_matches)
print("final targets equal extra source tokens:", final_targets_match_source)
print("shifted input values:")
print(input_batch[:, 1:])
print("target prefix values:")
print(target_batch[:, :-1])

target prefixes equal shifted inputs: True
final targets equal extra source tokens: True
shifted input values:
tensor([[ 7,  5,  0,  3,  2, 13,  0],
        [ 0, 10,  9,  0, 13,  7,  5]])
target prefix values:
tensor([[ 7,  5,  0,  3,  2, 13,  0],
        [ 0, 10,  9,  0, 13,  7,  5]])


The local overlap check catches most shift mistakes, while the final-token check confirms the extra future token came from the correct source index.


## Sample start indexes

Training commonly samples window starts rather than choosing them by hand.

Sampling with replacement means the same source window may appear more than once in a batch.

A dedicated generator gives this example deterministic random starts without changing global random state.


In [8]:
def sample_gpt_batch(
    token_ids: torch.Tensor,
    batch_size: int,
    context_length: int,
    generator: torch.Generator | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if batch_size < 1:
        raise ValueError("batch_size must be positive.")

    number_of_valid_starts = token_ids.numel() - context_length
    if number_of_valid_starts < 1:
        raise ValueError("token_ids must be longer than context_length.")

    start_indexes = torch.randint(
        low=0,
        high=number_of_valid_starts,
        size=(batch_size,),
        generator=generator,
        device=token_ids.device,
    )
    input_batch, target_batch = build_gpt_batch(
        token_ids, start_indexes, context_length
    )
    return input_batch, target_batch, start_indexes


random_generator = torch.Generator(device=device).manual_seed(69)
batch_size = 4
input_batch, target_batch, sampled_starts = sample_gpt_batch(
    all_token_ids,
    batch_size=batch_size,
    context_length=context_length,
    generator=random_generator,
)

print("sampled starts:", sampled_starts.tolist())
print("input shape:", input_batch.shape)
print("target shape:", target_batch.shape)
print("batch | input | target")
print("-" * 46)
for batch_index in range(batch_size):
    decoded_input = decode_token_ids(input_batch[batch_index].tolist(), id_to_character)
    decoded_target = decode_token_ids(
        target_batch[batch_index].tolist(), id_to_character
    )
    print(f"{batch_index:>5} | {decoded_input!r:>12} | {decoded_target!r:>12}")

sampled starts: [54, 11, 9, 43]
input shape: torch.Size([4, 8])
target shape: torch.Size([4, 8])
batch | input | target
----------------------------------------------
    0 |   't ran to' |   ' ran to '
    1 |   ' on the ' |   'on the m'
    2 |   'at on th' |   't on the'
    3 |   'rug. the' |   'ug. the '


The sampled batch still satisfies the same deterministic slicing rule after its start indexes are chosen.


## Connect batches to model logits

Chapter 68's model returns logits with shape `[B, T, V]`.

Targets have shape `[B, T]`, so each logit vector aligns with one target token ID.

The next cell uses deterministic fake logits to isolate the loss-shape calculation from model architecture.


In [9]:
vocabulary_size = len(characters)
loss_generator = torch.Generator(device=device).manual_seed(69)
fake_vocabulary_logits = torch.randn(
    batch_size,
    context_length,
    vocabulary_size,
    generator=loss_generator,
    device=device,
)
flattened_logits = fake_vocabulary_logits.reshape(-1, vocabulary_size)
flattened_targets = target_batch.reshape(-1)
flattened_loss = torch.nn.functional.cross_entropy(flattened_logits, flattened_targets)
per_position_loss = torch.nn.functional.cross_entropy(
    fake_vocabulary_logits.transpose(1, 2),
    target_batch,
    reduction="none",
)

print("logits shape:", fake_vocabulary_logits.shape)
print("targets shape:", target_batch.shape)
print("flattened logits shape:", flattened_logits.shape)
print("flattened targets shape:", flattened_targets.shape)
print("per-position loss shape:", per_position_loss.shape)
print("flattened loss:", flattened_loss.item())
print("mean position loss:", per_position_loss.mean().item())
print("loss calculations match:", end=" ")
print(torch.allclose(flattened_loss, per_position_loss.mean()))

logits shape: torch.Size([4, 8, 15])
targets shape: torch.Size([4, 8])
flattened logits shape: torch.Size([32, 15])
flattened targets shape: torch.Size([32])
per-position loss shape: torch.Size([4, 8])
flattened loss: 3.267484664916992
mean position loss: 3.2674849033355713
loss calculations match: True


The scalar loss is the mean of all 32 entries in the `[4, 8]` per-position loss tensor.

Using `reshape` makes the intended layout explicit without requiring the source tensor to be contiguous.


## Map flattened rows back to positions

Flattening rearranges the batch-position grid but does not discard any prediction.

For row-major flattening, `flat_index = batch_index × T + position`.


In [10]:
print("flat | batch | position | target ID | target token | loss")
print("-" * 66)
for flat_index in range(10):
    batch_index = flat_index // context_length
    position = flat_index % context_length
    target_id = flattened_targets[flat_index].item()
    target_token = id_to_character[target_id]
    position_loss = per_position_loss[batch_index, position].item()
    print(
        f"{flat_index:>4} | "
        f"{batch_index:>5} | "
        f"{position:>8} | "
        f"{target_id:>9} | "
        f"{target_token!r:>12} | "
        f"{position_loss:>6.3f}"
    )

flat | batch | position | target ID | target token | loss
------------------------------------------------------------------
   0 |     0 |        0 |         0 |          ' ' |  3.523
   1 |     0 |        1 |        11 |          'r' |  1.758
   2 |     0 |        2 |         2 |          'a' |  3.579
   3 |     0 |        3 |         9 |          'n' |  2.440
   4 |     0 |        4 |         0 |          ' ' |  4.331
   5 |     0 |        5 |        13 |          't' |  3.612
   6 |     0 |        6 |        10 |          'o' |  2.624
   7 |     0 |        7 |         0 |          ' ' |  2.045
   8 |     1 |        0 |        10 |          'o' |  2.465
   9 |     1 |        1 |         9 |          'n' |  3.376


Every flattened row corresponds to exactly one cell in the original `[B, T]` target grid.


## All-position loss versus final-position loss

Using all positions provides `B × T` supervised predictions per batch.

Using only the final position would provide only `B` predictions and would discard valid training signal from the earlier positions.

The two losses are both mathematically defined, but standard next-token GPT training normally uses all available positions.


In [11]:
final_position_loss = torch.nn.functional.cross_entropy(
    fake_vocabulary_logits[:, -1, :],
    target_batch[:, -1],
)

print("all-position prediction count:", batch_size * context_length)
print("final-only prediction count:", batch_size)
print("all-position loss:", flattened_loss.item())
print("final-position-only loss:", final_position_loss.item())

all-position prediction count: 32
final-only prediction count: 4
all-position loss: 3.267484664916992
final-position-only loss: 2.9371910095214844


Generation often consumes only final-position logits, but that does not imply training should ignore earlier positions.


## Why causal masking is still necessary

The complete input row is present in memory during a parallel training forward pass.

Without a causal mask, an early position could read later input tokens that include its answer.

For example, input position `0` in `the cat ` should predict `h`, which is already stored at input position `1`.

The target shift defines what each position should predict.

The causal mask defines which input positions each prediction may use.

Both are required for valid parallel next-token training.


## Diagnose off-by-one mistakes

Two common wrong slices can be identified before loss computation.

Using the input slice again creates copying targets, while omitting the final `+ 1` creates a target that is one token too short.


In [12]:
diagnostic_start = 0
correct_input = all_token_ids[diagnostic_start : diagnostic_start + context_length]
copying_target = all_token_ids[diagnostic_start : diagnostic_start + context_length]
too_short_target = all_token_ids[
    diagnostic_start + 1 : diagnostic_start + context_length
]
correct_target = all_token_ids[
    diagnostic_start + 1 : diagnostic_start + context_length + 1
]

print("input length:", correct_input.numel())
print("copying target equals input:", torch.equal(copying_target, correct_input))
print("too-short target length:", too_short_target.numel())
print("correct target length:", correct_target.numel())
print("correct local shift:", end=" ")
print(torch.equal(correct_target[:-1], correct_input[1:]))

input length: 8
copying target equals input: True
too-short target length: 7
correct target length: 8
correct local shift: True


Shape equality alone cannot catch copying targets, so the local shift invariant is the stronger check.


## Shape summary

| Value | Shape | Meaning |
|---|---|---|
| Source token sequence | `[N]` | Encoded training text |
| Start indexes | `[B]` | One sampled source start per window |
| Input batch | `[B, T]` | Tokens visible to the model |
| Target batch | `[B, T]` | Next token for every input position |
| Model logits | `[B, T, V]` | Raw vocabulary scores at every position |
| Per-position loss | `[B, T]` | One cross-entropy value per target |
| Flattened logits | `[B × T, V]` | Logits arranged for cross-entropy |
| Flattened targets | `[B × T]` | Target IDs arranged for cross-entropy |


## Common mistakes

- Do not use the same slice for inputs and targets.
- Do not omit the final `+ 1` from the target slice stop.
- Do not sample `start_index = N - T`, because its target would be too short.
- Do not expect sampled start indexes to be unique when sampling with replacement.
- Do not use only one target per sequence when the goal is all-position GPT training.
- Do not confuse target shifting with causal masking.
- Do not apply softmax before passing logits to `cross_entropy`.


## Takeaways

GPT next-token targets use the same source text shifted forward by one token.

```python
input_row = token_ids[start_index : start_index + context_length]
target_row = token_ids[
    start_index + 1 : start_index + context_length + 1
]
```

Input and target batches both have shape `[B, T]`.

A useful invariant is `target_batch[:, :-1] == input_batch[:, 1:]`.

The model's `[B, T, V]` logits provide one vocabulary prediction for every target token.

> A batch has `B` sequence windows and `B × T` supervised next-token positions.

Causal masking prevents those parallel predictions from reading future input tokens.


## What comes next

The next chapter will feed these sampled input-target batches into `TinyGPT`, backpropagate the all-position loss, and update the model with an optimizer.

That training loop will connect correct batch construction to measurable learning.
